### Read Bronze Reviews

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.reviews"
SILVER_TABLE = "olist_mlops.silver.reviews_clean"

bronze_reviews = spark.table(BRONZE_TABLE)

print("Bronze review rows:", bronze_reviews.count())

bronze_reviews.printSchema()

display(bronze_reviews.limit(5))

### Validate data types before conversion
- Validate conversions

In [0]:
type_mapping = {
    "review_score": "INT",
    "review_creation_date": "TIMESTAMP_NTZ",
    "review_answer_timestamp": "TIMESTAMP_NTZ"
}

for column, target_type in type_mapping.items():

    failed_casts = bronze_reviews.filter(
        F.col(column).isNotNull()
        & F.expr(
            f"try_cast(`{column}` AS {target_type})"
        ).isNull()
    ).count()

    print(f"{column}: {failed_casts} failed casts")

    assert failed_casts == 0, (
        f"Invalid conversion in {column}"
    )

print("REVIEW TYPE VALIDATION PASSED")

### Transform Review attributes
- Apply correct types and quality flags

In [0]:
reviews_clean = (
    bronze_reviews

    .withColumn(
        "review_score",
        F.expr("try_cast(review_score AS INT)")
    )

    .withColumn(
        "review_creation_date",
        F.expr(
            "try_cast(review_creation_date AS TIMESTAMP_NTZ)"
        )
    )

    .withColumn(
        "review_answer_timestamp",
        F.expr(
            "try_cast(review_answer_timestamp AS TIMESTAMP_NTZ)"
        )
    )

    # Preserve the original review text.
    # Only create flags for text availability.

    .withColumn(
        "has_review_title",
        F.length(
            F.trim(
                F.coalesce(
                    F.col("review_comment_title"),
                    F.lit("")
                )
            )
        ) > 0
    )

    .withColumn(
        "has_review_message",
        F.length(
            F.trim(
                F.coalesce(
                    F.col("review_comment_message"),
                    F.lit("")
                )
            )
        ) > 0
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

reviews_clean.printSchema()

### Validate review scores
- Review score distribution

In [0]:
display(
    reviews_clean
    .groupBy("review_score")
    .count()
    .orderBy("review_score")
)

invalid_scores = reviews_clean.filter(
    F.col("review_score").isNull()
    | ~F.col("review_score").between(1, 5)
).count()

print("Invalid review scores:", invalid_scores)

assert invalid_scores == 0

print("REVIEW SCORE VALIDATION PASSED")

### Validate composite primary keys
- Validate key integrity

In [0]:
source_count = bronze_reviews.count()
silver_count = reviews_clean.count()

null_keys = reviews_clean.filter(
    F.col("review_id").isNull()
    | F.col("order_id").isNull()
).count()

duplicate_keys = (
    reviews_clean
    .groupBy("review_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Bronze rows:", source_count)
print("Silver rows:", silver_count)
print("Null composite keys:", null_keys)
print("Duplicate composite keys:", duplicate_keys)

assert source_count == silver_count == 99224
assert null_keys == 0
assert duplicate_keys == 0

print("REVIEW KEY VALIDATION PASSED")

### Validate optional comments
- Check text completeness

In [0]:
text_quality = reviews_clean.agg(

    F.count(
        F.when(
            F.col("review_comment_title").isNull(),
            1
        )
    ).alias("missing_titles"),

    F.count(
        F.when(
            F.col("review_comment_message").isNull(),
            1
        )
    ).alias("missing_messages"),

    F.count(
        F.when(
            F.col("has_review_title"),
            1
        )
    ).alias("reviews_with_title"),

    F.count(
        F.when(
            F.col("has_review_message"),
            1
        )
    ).alias("reviews_with_message")
)

display(text_quality)

### Validate timestamp consistency
- Timestamp checks

In [0]:
timestamp_quality = reviews_clean.agg(

    F.count(
        F.when(
            F.col("review_creation_date").isNull(),
            1
        )
    ).alias("missing_creation_date"),

    F.count(
        F.when(
            F.col("review_answer_timestamp").isNull(),
            1
        )
    ).alias("missing_answer_timestamp"),

    F.count(
        F.when(
            F.col("review_answer_timestamp")
            < F.col("review_creation_date"),
            1
        )
    ).alias("answer_before_creation")
)

display(timestamp_quality)

### Validate relationship with Orders
- Foreign-key validation

In [0]:
order_keys = (
    spark.table("olist_mlops.silver.orders_clean")
    .select("order_id")
    .distinct()
)

orphan_reviews = (
    reviews_clean
    .select("order_id")
    .distinct()
    .join(
        order_keys,
        on="order_id",
        how="left_anti"
    )
    .count()
)

print("Orphan review order IDs:", orphan_reviews)

assert orphan_reviews == 0

print("REVIEW REFERENTIAL INTEGRITY PASSED")

### Save Reviews as a Silver Delta table
- Write the table

In [0]:
(
    reviews_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

written_count = spark.table(SILVER_TABLE).count()

assert written_count == 99224

print("Created:", SILVER_TABLE)
print("Rows:", written_count)

### Analyze review behaviour with SQL
- Review score distribution

In [0]:
%sql

SELECT
    review_score,
    COUNT(*) AS review_records,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM olist_mlops.silver.reviews_clean
GROUP BY review_score
ORDER BY review_score;

- Review text availability

In [0]:
%sql

SELECT
    has_review_title,
    has_review_message,
    COUNT(*) AS review_count
FROM olist_mlops.silver.reviews_clean
GROUP BY has_review_title, has_review_message
ORDER BY has_review_title, has_review_message;

- Orders with multiple reviews

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS review_count,
    COUNT(DISTINCT review_id) AS distinct_review_ids
FROM olist_mlops.silver.reviews_clean
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY review_count DESC, order_id
LIMIT 10;